# Extract & Label Chess Piece Glyphs from PDF

**Workflow:**
1. Load previous classifier (optional) to pre-filter candidates
2. Extract candidate glyphs from PDF
3. Manually label ambiguous ones
4. Train new classifier
5. Download zip with labeled glyphs + classifier

## Step 1 — Mount Google Drive and load PDF

In [ ]:
from google.colab import drive
import os

drive.mount('/content/gdrive')

In [ ]:
PDF_PATH = '/content/gdrive/MyDrive/chess_book.pdf'

if not os.path.exists(PDF_PATH):
    print(f'❌ PDF not found: {PDF_PATH}')
else:
    size_mb = os.path.getsize(PDF_PATH) / (1024*1024)
    print(f'✅ PDF loaded: {PDF_PATH}  ({size_mb:.1f} MB)')

## Step 2 — Install dependencies

In [ ]:
!apt-get install -y poppler-utils
!pip install -q pdfplumber pdf2image pillow scikit-learn scikit-image

## Step 3 — Configuration

In [ ]:
import pdfplumber
from pdf2image import convert_from_path
from PIL import Image
import os
from pathlib import Path
import pickle
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from skimage import filters, transform
import zipfile

START_PAGE = 1
END_PAGE = 20
GLYPH_DPI = 150
IMG_SIZE = 32
CONFIDENCE_THRESHOLD = 0.7

GLYPHS_DIR = './glyphs'
PIECE_CLASSES = ['K', 'Q', 'R', 'B', 'N']

Path(GLYPHS_DIR).mkdir(exist_ok=True)
for piece in PIECE_CLASSES:
    Path(f'{GLYPHS_DIR}/{piece}').mkdir(exist_ok=True)

print(f'✅ Output directory: {GLYPHS_DIR}/')
print(f'   Subdirectories for: {" ".join(PIECE_CLASSES)}')

## Step 3.5 — Load previous classifier (optional)

In [ ]:
from google.colab import files

classifier = None
previous_labeled_count = {piece: 0 for piece in PIECE_CLASSES}

print('📁 Upload a previous classifier zip (optional, press Skip if none):')
print('   This will use the learned classifier to pre-filter candidates')
print()

try:
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('./previous')
            
            if os.path.exists('./previous/classifier.pkl'):
                with open('./previous/classifier.pkl', 'rb') as f:
                    classifier = pickle.load(f)
                print(f'✅ Loaded previous classifier')
            
            for piece in PIECE_CLASSES:
                path = f'./previous/glyphs/{piece}'
                if os.path.exists(path):
                    count = len([f for f in os.listdir(path) if f.endswith('.png')])
                    previous_labeled_count[piece] = count
            
            total_prev = sum(previous_labeled_count.values())
            print(f'✅ Found {total_prev} previously labeled glyphs')
except:
    print('ℹ️  No previous classifier uploaded')

## Step 4 — Extract features and filter candidates

In [ ]:
def extract_image_features(img):
    if img.width < 5 or img.height < 5:
        return None
    
    img_resized = transform.resize(np.array(img), (IMG_SIZE, IMG_SIZE), anti_aliasing=True)
    
    features = []
    features.append(np.mean(img_resized))
    features.append(np.std(img_resized))
    features.append(img.width / max(img.height, 1))
    
    edges = filters.sobel(img_resized)
    features.append(np.mean(edges))
    
    features.append(np.mean(np.sum(img_resized, axis=0)))
    features.append(np.mean(np.sum(img_resized, axis=1)))
    
    return np.array(features)

def has_glyph_character(text):
    for char in text:
        if char in '♔♕♖♗♘♙♚♛♜♝♞♟':
            return True
        if ord(char) > 255:
            return True
    return False

def render_page(pdf_path, page_num, dpi=150):
    images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1, dpi=dpi)
    return images[0] if images else None

glyph_words = []

with pdfplumber.open(PDF_PATH) as pdf:
    pdf_page_count = len(pdf.pages)
    end_page = min(END_PAGE, pdf_page_count)
    
    for page_idx in range(START_PAGE - 1, end_page):
        page_num = page_idx + 1
        pdf_page = pdf.pages[page_idx]
        
        try:
            words = pdf_page.extract_words()
        except:
            continue
        
        page_image = render_page(PDF_PATH, page_idx, dpi=GLYPH_DPI)
        if page_image is None:
            continue
        
        for word in words:
            text = word.get('text', '')
            
            if has_glyph_character(text):
                bbox = (word['x0'], word['top'], word['x1'], word['bottom'])
                
                scale = GLYPH_DPI / 72.0
                x0 = max(0, int(bbox[0] * scale))
                y0 = max(0, int(bbox[1] * scale))
                x1 = min(page_image.width, int(bbox[2] * scale))
                y1 = min(page_image.height, int(bbox[3] * scale))
                
                if x1 <= x0 or y1 <= y0:
                    continue
                
                crop = page_image.crop((x0, y0, x1, y1))
                features = extract_image_features(crop)
                
                if features is None:
                    continue
                
                glyph_words.append({
                    'page': page_num,
                    'text': text,
                    'bbox': bbox,
                    'crop': crop,
                    'features': features,
                })

print(f'✅ Found {len(glyph_words)} candidate words with glyphs')

if classifier is not None:
    filtered_words = []
    for word in glyph_words:
        conf = np.max(classifier.predict_proba([word['features']])[0])
        word['confidence'] = conf
        if conf >= CONFIDENCE_THRESHOLD:
            filtered_words.append(word)
    
    print(f'⚡ Classifier pre-filtered to {len(filtered_words)} candidates')
    glyph_words = filtered_words
else:
    print('ℹ️  No classifier available; showing all candidates')

## Step 5 — Manual labeling UI

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

current_idx = 0
saved_count = {piece: previous_labeled_count[piece] for piece in PIECE_CLASSES}
skipped = 0
discarded = 0

def show_glyph(idx):
    global current_idx, saved_count, skipped, discarded
    
    if idx >= len(glyph_words):
        clear_output()
        print(f'✅ Labeling complete!')
        print(f'\nSaved:')
        for piece in PIECE_CLASSES:
            print(f'  {piece}: {saved_count[piece]}')
        print(f'  Skipped: {skipped}')
        print(f'  Discarded: {discarded}')
        return
    
    current_idx = idx
    word_info = glyph_words[idx]
    crop = word_info['crop']
    
    clear_output()
    conf_text = f" (confidence: {word_info.get('confidence', 0):.2%})" if 'confidence' in word_info else ""
    print(f'Glyph {idx + 1}/{len(glyph_words)} — P{word_info["page"]}: "{word_info["text"]}"{conf_text}')
    print()
    display(crop)
    print()
    
    def save_glyph(piece):
        count = saved_count[piece]
        filename = f'{GLYPHS_DIR}/{piece}/{count + 1:04d}.png'
        crop.save(filename)
        saved_count[piece] += 1
        show_glyph(idx + 1)
    
    def skip():
        global skipped
        skipped += 1
        show_glyph(idx + 1)
    
    def discard():
        global discarded
        discarded += 1
        show_glyph(idx + 1)
    
    buttons = [
        widgets.Button(description='K (King)', button_style='info'),
        widgets.Button(description='Q (Queen)', button_style='info'),
        widgets.Button(description='R (Rook)', button_style='info'),
        widgets.Button(description='B (Bishop)', button_style='info'),
        widgets.Button(description='N (Knight)', button_style='info'),
        widgets.Button(description='Skip', button_style='warning'),
        widgets.Button(description='❌ Discard', button_style='danger'),
    ]
    
    buttons[0].on_click(lambda _: save_glyph('K'))
    buttons[1].on_click(lambda _: save_glyph('Q'))
    buttons[2].on_click(lambda _: save_glyph('R'))
    buttons[3].on_click(lambda _: save_glyph('B'))
    buttons[4].on_click(lambda _: save_glyph('N'))
    buttons[5].on_click(lambda _: skip())
    buttons[6].on_click(lambda _: discard())
    
    display(widgets.HBox(buttons))

if len(glyph_words) > 0:
    show_glyph(0)
else:
    print('❌ No candidates to label')

## Step 6 — Train classifier

In [ ]:
X_train = []
y_train = []

for piece_idx, piece in enumerate(PIECE_CLASSES):
    path = f'{GLYPHS_DIR}/{piece}'
    if os.path.exists(path):
        for img_file in os.listdir(path):
            if img_file.endswith('.png'):
                img = Image.open(f'{path}/{img_file}').convert('L')
                features = extract_image_features(img)
                if features is not None:
                    X_train.append(features)
                    y_train.append(piece_idx)

if len(X_train) > 10:
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    
    new_classifier = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=10)
    new_classifier.fit(X_train, y_train)
    
    with open(f'{GLYPHS_DIR}/classifier.pkl', 'wb') as f:
        pickle.dump(new_classifier, f)
    
    print(f'✅ Trained classifier on {len(X_train)} labeled glyphs')
    print(f'   Accuracy: {new_classifier.score(X_train, y_train):.1%}')
else:
    print('⚠️  Not enough labeled samples (need ≥10)')

## Step 7 — Export zip with classifier

In [ ]:
import shutil

zip_filename = 'chess_glyphs_classifier.zip'
shutil.make_archive('chess_glyphs_classifier', 'zip', '.', GLYPHS_DIR)

final_counts = {}
total_glyphs = 0
for piece in PIECE_CLASSES:
    path = f'{GLYPHS_DIR}/{piece}'
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.endswith('.png')])
        final_counts[piece] = count
        total_glyphs += count

print(f'✅ Export complete!')
print(f'\n📦 Zip file: {zip_filename}')
print(f'\nContents:')
print(f'  glyphs/classifier.pkl')
for piece in PIECE_CLASSES:
    count = final_counts.get(piece, 0)
    print(f'  glyphs/{piece}/ ({count} images)')
print(f'\n  TOTAL: {total_glyphs} labeled glyphs')
print(f'\n💾 Download and upload to this notebook in the next session')

from google.colab import files
files.download(zip_filename)